In [11]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "bourjade2014bonobos")
original_data_pathway = os.path.join(pathway, "original_data")

# complete_path_1 = os.path.join(original_data_pathway, "Data future planning for stats_all_species.csv")
complete_path_2 = os.path.join(original_data_pathway, "Raw data ape future planning_bonobos.csv")
complete_path_3 = os.path.join(original_data_pathway, "Raw data ape future planning_chimpanzees.csv")
complete_path_4 = os.path.join(original_data_pathway, "Raw data ape future planning_chimpanzees_follow_up.csv")
complete_path_5 = os.path.join(original_data_pathway, "Raw data ape future planning_orangs.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [12]:
import pandas as pd
import numpy as np
import pyreadstat

# df1 = pd.read_csv(complete_path_1)
# df1['experiment_name'] = "stats"
# df1['experiment'] = "4"
df2 = pd.read_csv(complete_path_2)
df3 = pd.read_csv(complete_path_3)
df5 = pd.read_csv(complete_path_5)

experiment_import = [[df2, 'raw', '2', '2009'],
                    [df3, 'raw','3','2010'],
                    [df5, 'raw','1','2007']]

for x, y, k, b in experiment_import:
    x['experiment_name'] = y
    x['experiment'] = k
    x['year'] = b

df4 = pd.read_csv(complete_path_4)
df4['experiment_name'] = "follow_up"
df4['experiment'] = "3"
df4[['day','month', 'year']] = df4['Date'].str.split('/',expand=True)



In [13]:
data_frames=[ df2, df3, df4, df5]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape",
        'individu':'ape',
        "collectv":"collect_v_ab",
        "collectnv":"collect_nv_ab",
        "valuable tokens ab":"collect_v_ab",
        "valuable tokens ba":"collect_v_ba",
        "no-valuable tokens ab":"collect_nv_ab",
        "no-valuable tokens ba":"collect_nv_ba",
        'nonvaluable':"collect_nv_ba",
        'valuable':"collect_v_ba",
        }, inplace=True)
    x['study_id']="bourjade2014bonobos"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [14]:
comp_path_name_errors = os.path.join(pathway_gen, "bourjade_name_errors.csv")


df_name  = pd.read_csv(comp_path_name_errors)
df_name.columns = map(str.lower, df_name.columns)
df_name=df_name.applymap(lambda s: s.lower() if type(s) == str else s)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

In [15]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

In [16]:
fulldf.rename(columns={"test number": "test_number",
    # "collectv":"collect_v",
    # "collectnv":"collect_nv",
    "collecttot":"collect_tot",
    "presv":"pres_v",
    "cubeab":"cube_ab",
    "tubeab":"tube_ab",
    "chainab":"chain_ab",
    "cubeba":"cube_ba",
    "tubeba":"tube_ba",
    "chainba":"chain_ba",
    "token rwd":"token_rwd",
    # "valuable tokens ab":"valuable_tokens_ab",
    # "valuable tokens ba":"valuable_tokens_ba",
    # "no-valuable tokens ab":"no-valuable_tokens_ab",
    # "no-valuable tokens ba":"no-valuable_tokens_ba",
    "collection a->b  tb":"collection_a-to_b__tb",
    "collection a->b  cb":"collection_a-to_b__cb",
    "collection a->b  ch":"collection_a-to_b__ch",
    "transport b->a    tb":"transport_b-to_a__tb",
    "transport b->a    cb":"transport_b-to_a__cb",
    "transport b->a    ch":"transport_b-to_a__ch",
    "exchange   tb":"exchange__tb",
    "exchange  cb":"exchange__cb",
    "exchange   ch":"exchange __ch"}, inplace=True)
fulldf.columns
fulldf.dropna(subset=['ape'], inplace=True)

fulldf.rename(columns={"ape": "participant"}, inplace=True)

In [17]:
reward_tubes = [['joey','tube'],
                 ['limbuko','tube'],
                 ['ulindi','tube'],
                ['dokana','tube'],
                ['pini','tube'],
                 ['gertrudia', 'tb'],
                 ['alexandra', 'tb'],
                 ['kuno', 'cube'],
                 ['jahaba','cube'],
                 ['bimbo', 'cube'],
                 ['padana','cube'],
                 ['yasa','cube'],
                 ['jahaga','cb']]
for x, y in reward_tubes:
    fulldf.loc[fulldf.participant == x, ['token_rwd']] = y


In [18]:
fulldf = fulldf[~fulldf.experiment_name.str.contains("follow_up")]



In [19]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age": "age_in_years"}, inplace=True)

In [20]:

fulldf=fulldf[['study_id','experiment',  'year',
       'participant','age_in_years', 'sex', 'species', 'condition', 'test', 
       'collect_v_ab','collect_nv_ab', 'collect_v_ba', 'collect_nv_ba','token_rwd']]
##removed 'date','block', 'collect_tot', 'retourtot', 'pres_v'
##removed empty columns 'experiment_name',  'month', 'day','hour',
## removed as suggested by author in glossary: 'test_number','comments'
##removed as suggested by author 'cube_ab', 'tube_ab','chain_ab', 'cube_ba', 'tube_ba', 'chain_ba','collection_a-to_b__tb', 'collection_a-to_b__cb', 'collection_a-to_b__ch', 'transport_b-to_a__tb', 'transport_b-to_a__cb','transport_b-to_a__ch', 'exchange__tb', 'exchange__cb', 'exchange __ch',

In [21]:
comp_out_path_stand = os.path.join(out_pathway, 'bourjade2014bonobos_standardized.csv')
fulldf.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names = fulldf.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'bourjade2014bonobos_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

In [22]:
# for index in range(1,5):
#     exp = fulldf[fulldf['experiment'] == str(index)]
#     exp = exp.dropna(axis=1, how='all')
#     comp_out_path = os.path.join(out_pathway, 'bourjade2014bonobos_exp'+str(index)+'_standardized.csv')
#     exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
#     names = exp.columns.tolist()
#     exp_g = pd.DataFrame(names)
#     exp_g = exp_g.rename(columns={0: "column_name"})
#     exp_g["description"] = ""
#     exp_g=exp_g[["column_name", "description"]]
#     comp_out_path_glossary = os.path.join(out_pathway, 'bourjade2014bonobos_exp'+str(index)+'_glossary.csv')
#     exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
